# DATA

## EXTRACCION DE DATOS
Para el desarrollo de este proyecto se recurrió a dos tipos de fuentes. Por un lado, las pistas de audio utilizadas para construir el dataset fueron obtenidas de Free Music Archive (https://freemusicarchive.org), plataforma de distribución musical que ofrece canciones bajo licencias Creative Commons, permitiendo su uso libre en contextos académicos e investigativos sin restricciones de derechos de autor. Las canciones fueron seleccionadas y organizadas manualmente por género musical, garantizando un mínimo de 50 pistas por categoría.

## PROCESAMIENTO Y MODELAMIENTO DE LOS DATOS

### Librerias 
Librosa: es una librería especializada en análisis de audio y música que permite cargar archivos de sonido y extraer características acústicas. Es el núcleo del proceso de extracción de features en este proyecto.

NumPy: es la librería fundamental para computación numérica en Python. Proporciona estructuras de datos tipo array de alto rendimiento y operaciones matemáticas vectorizadas, siendo la base sobre la que trabajan la mayoría de librerías científicas incluyendo Librosa.

Pandas es la librería estándar para manipulación y análisis de datos tabulares en Python. Permite construir, limpiar y transformar dataframes, que es la estructura utilizada para almacenar y exportar el dataset final en formato CSV.

Os es un módulo nativo de Python que permite interactuar con el sistema operativo, específicamente con el sistema de archivos. En este proyecto se utiliza para recorrer las carpetas de géneros, listar los archivos de audio disponibles y construir las rutas de acceso a cada canción.

In [6]:
import librosa
import numpy as np
import pandas as pd
import os

### Extracción de Features 
El código implementa un pipeline de extracción automática de características acústicas a partir de archivos de audio organizados por género musical. Para cada canción se carga la pista completa y se extraen tres segmentos de 30 segundos correspondientes al inicio, la mitad y el final de la pista. De cada segmento se calculan 27 features: siete características acústicas globales (tempo, spectral centroid, spectral bandwidth, rolloff, zero crossing rate, RMS energy y chroma STFT) y 20 coeficientes MFCC que representan el timbre de la señal. Cada segmento procesado queda registrado como una fila independiente en el dataset, junto con su etiqueta de género y el nombre del archivo de origen. El resultado final es un archivo CSV estructurado y listo para la etapa de modelado.

In [ ]:
def extraerfeatures(y, sr): # Extrae características de un segmento de audio
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features = {
        'tempo':              librosa.beat.tempo(y=y, sr=sr)[0],
        'spectral_centroid':  librosa.feature.spectral_centroid(y=y, sr=sr).mean(),
        'spectral_bandwidth': librosa.feature.spectral_bandwidth(y=y, sr=sr).mean(),
        'rolloff':            librosa.feature.spectral_rolloff(y=y, sr=sr).mean(),
        'zero_crossing_rate': librosa.feature.zero_crossing_rate(y).mean(),
        'rms':                librosa.feature.rms(y=y).mean(),
        'chroma_stft':        librosa.feature.chroma_stft(y=y, sr=sr).mean(),
    }
    for i, coef in enumerate(mfccs):
        features[f'mfcc{i+1}'] = coef.mean()
    return features

def obtenersegmentos(duracion_total):
    inicio = 0
    mitad  = int(duracion_total / 2) - 15
    final  = int(duracion_total) - 30
    return {'inicio': inicio, 'mitad': mitad, 'final': final}

filas = []
carpeta_raiz = '' # Ruta de la carpeta raiz que contiene las subcarpetas de géneros musicales

# Diagnóstico inicial
print(f"Carpeta raíz existe: {os.path.exists(carpeta_raiz)}")
print(f"Contenido de la carpeta raíz: {os.listdir(carpeta_raiz)}")

for genero in os.listdir(carpeta_raiz):
    carpeta_genero = os.path.join(carpeta_raiz, genero)
    if not os.path.isdir(carpeta_genero):
        print(f"  SKIP (no es carpeta): {genero}")
        continue

    archivos_genero = os.listdir(carpeta_genero)
    print(f"\n{genero}: {len(archivos_genero)} archivos → {archivos_genero[:3]}")  # muestra los primeros 3

    for archivo in archivos_genero:
        if not (archivo.endswith('.mp3') or archivo.endswith('.wav')):
            print(f"  SKIP (formato no válido): {archivo}")
            continue

        ruta = os.path.join(carpeta_genero, archivo)
        try:
            cancion, sr = librosa.load(ruta, sr=None)
            duracion_total = librosa.get_duration(y=cancion, sr=sr)

            if duracion_total < 90:
                print(f"  SKIP (muy corta {duracion_total:.0f}s): {archivo}")
                continue

            segmentos = obtenersegmentos(duracion_total)

            for nombre_seg, seg_inicio in segmentos.items():
                inicio_muestra = int(seg_inicio * sr)
                fin_muestra    = int((seg_inicio + 30) * sr)
                segmento       = cancion[inicio_muestra:fin_muestra]

                features = extraerfeatures(segmento, sr)
                features['label']    = genero
                features['cancion']  = archivo
                features['segmento'] = nombre_seg
                filas.append(features)

            print(f"  OK: {archivo} ({duracion_total:.0f}s)")

        except Exception as e:
            print(f"  ERROR en {archivo}: {e}")

# Verificación antes de guardar
if len(filas) == 0:
    print("\nERROR: no se procesó ningún archivo. Revisa los SKIP y ERROR arriba.")
else:
    df = pd.DataFrame(filas)
    df.to_csv('dataset_musical.csv', index=False)
    print(f"\nDataset guardado: {df.shape}")
    print(f"Muestras por género:\n{df['label'].value_counts()}")

Carpeta raíz existe: True
Contenido de la carpeta raíz: ['Rock']

Rock: 5 archivos → ['Here Comes A Big Black Cloud!! - The Fly Pt. II.mp3', 'Rod - Come Back (and stop those rainy days).mp3', 'Rod - Inner Rhythm.mp3']


C:\Users\Usuario\AppData\Local\Temp\ipykernel_12516\3566351819.py:4: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  'tempo':              librosa.beat.tempo(y=y, sr=sr)[0],


  OK: Here Comes A Big Black Cloud!! - The Fly Pt. II.mp3 (144s)
  OK: Rod - Come Back (and stop those rainy days).mp3 (1390s)
  OK: Rod - Inner Rhythm.mp3 (1402s)
  OK: Rod - Inner Voices.mp3 (464s)
  OK: Rod - Sursaut.mp3 (150s)

Dataset guardado: (15, 30)
Muestras por género:
label
Rock    15
Name: count, dtype: int64


## RESULTADOS

In [7]:
pd.read_csv('dataset_musical.csv').head()

,tempo,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,rms,chroma_stft,mfcc1,mfcc2,mfcc3,...,mfcc14,mfcc15,mfcc16,mfcc17,mfcc18,mfcc19,mfcc20,label,cancion,segmento
0,120.185320,1637.230677,1835.305974,3155.155921,0.043583,0.390755,0.444349,-80.520620,215.48050,-60.879402,...,3.418576,-3.260364,1.338831,-3.248235,-3.789156,-4.729334,-4.885305,Rock,Here Comes A Big Black Cloud!! - The Fly Pt. I...,inicio
1,123.046875,1768.673633,1870.426939,3376.287917,0.049069,0.452624,0.460819,-33.998665,224.60306,-84.549970,...,6.409605,-5.672055,5.314235,-3.091715,-0.902965,0.201558,-4.557393,Rock,Here Comes A Big Black Cloud!! - The Fly Pt. I...,mitad
2,120.185320,1720.919551,1805.266687,3282.171820,0.047967,0.317747,0.464138,-111.956800,191.59705,-66.413340,...,1.884634,-5.700083,1.337004,-6.333124,-3.881602,-4.471412,-3.517199,Rock,Here Comes A Big Black Cloud!! - The Fly Pt. I...,final
3,123.046875,2992.331118,3580.753736,5230.643371,0.081938,0.054848,0.443714,-304.928800,129.44681,-23.118452,...,7.078043,0.434119,3.629564,-1.794189,3.085832,0.552300,-0.664529,Rock,Rod - Come Back (and stop those rainy days).mp3,inicio
4,129.199219,1946.960585,2544.524589,3464.354052,0.046861,0.322928,0.460606,-79.577670,182.06450,-45.980110,...,-1.762941,-8.430030,-0.951798,-9.034123,-4.030103,-10.240243,-7.306660,Rock,Rod - Come Back (and stop those rainy days).mp3,mitad


Tempo
Mide los BPM (pulsaciones por minuto) de la canción. Librosa detecta los picos de energía que se repiten en el tiempo y calcula cada cuánto ocurren. Un género como el metal o el disco tiene tempo alto, el blues o el classical lo tiene bajo.

Spectral Centroid
Es el centro de gravedad del espectro de frecuencias. Se calcula como el promedio ponderado de todas las frecuencias presentes en la señal. Un valor alto significa que la energía está concentrada en frecuencias agudas (sonido brillante), uno bajo en frecuencias graves (sonido oscuro y profundo).

Spectral Bandwidth
Mide qué tan dispersas están las frecuencias alrededor del centroid. Si el ancho es grande, la canción tiene mucho contenido en un rango amplio de frecuencias. Si es pequeño, el sonido es más puro y concentrado. Géneros ruidosos como el metal tienen bandwidth alto.

Rolloff
Es la frecuencia por debajo de la cual se encuentra el 85% de la energía total de la señal. Sirve para detectar si una canción tiene mucho contenido en frecuencias altas o bajas. Distingue bien géneros con mucho agudo (rock, metal) de géneros más cálidos (jazz, blues).

Zero Crossing Rate
Cuenta cuántas veces por segundo la señal de audio pasa de un valor positivo a uno negativo. Una señal percusiva o ruidosa cruza el cero muy frecuentemente. Una señal tonal y melódica lo hace pocas veces. Es útil para separar géneros como hiphop y metal de classical y jazz.

RMS (Root Mean Square Energy)
Es la raíz cuadrada del promedio de los cuadrados de las amplitudes de la señal. Representa el volumen real promedio de la canción. Géneros intensos como metal tienen RMS alto, géneros acústicos o suaves como classical lo tienen bajo.

Chroma STFT
Mapea todas las frecuencias de la señal a las 12 notas musicales de la escala cromática (do, do#, re, re#...) y mide cuánto aparece cada una. Captura la armonía y la tonalidad de la canción. Es especialmente útil para distinguir géneros que tienen estructuras armónicas características, como el jazz o el reggae.

label
Indica el genero de la cancion

cancion
Indica el nombre del archivo

Segmento
Indica cual de los 3 segmentos representa la fila